<img src='https://www.icos-cp.eu/sites/default/files/2017-11/ICOS_CP_logo.png' width=400 align=right>


# STILT Plots

### Import modules

In [ ]:
#Import modules:
import numpy as np
import pandas as pd
import sys

#Import modules for spatiotemporal visualization:
import holoviews as hv
import geoviews as gv
import geoviews.feature as gf
import geoviews.tile_sources as gvts
from cartopy import crs
gv.extension('bokeh')

#Import STILT tools:
from icoscp.stilt import station

### Create STILT station object

In [ ]:
#Create STILT station obj:
st = station.get(id='KIT030')[0]
print(st)

### Get STILT time series

In [ ]:
start = '2018-01-01'
end = '2018-01-31'

st.get_ts(start, end)

### Plot STILT time series

In [ ]:
st.get_ts(start, end).plot(y=['co2.stilt', 'co2.background'], title=st.id, ylabel='ppm', figsize=(11,5))

### Get STILT footprints

In [ ]:
#Get footprint xarray:
stilt_fp = st.get_fp(start, end)

#View fp xarray:
stilt_fp

### Plot STILT footprints

In [ ]:
#Function to extract max absolute value from a tuple:
def return_max(val1, val2):
    
    if(abs(val1)>=abs(val2)):
        return abs(val1)
    
    else:
        return abs(val2)
    

#Set dimensions:
kdims = ['time', 'lon', 'lat']
vdims = ['foot']

#Get a tuple with min & max values for selected variable:
var_range = gv.Dataset(stilt_fp.foot.where(stilt_fp.foot>0.0, drop=True),
                       kdims=kdims, vdims=vdims).range('foot')

#Create geoviews dataset obj:
xr_dataset = gv.Dataset(stilt_fp.foot.where(stilt_fp.foot>0.0, drop=True),
                        kdims=kdims, vdims=vdims).redim.range(foot=(min(var_range),
                                                                    return_max(var_range[0],
                                                                               var_range[1])))

#Format dates:
hv.Dimension.type_formatters[np.datetime64] = '%Y-%m-%dT%H:%M:%S'

#Set location of slider:
hv.output(widget_location='bottom')

In [ ]:
%%opts Image [logz=True]

#Create plot:
xr_dataset.to(gv.Image, ['lon', 'lat'], dynamic=True).opts(cmap='viridis',colorbar=True, alpha=0.5, #cmap='gist_heat_r'
                                                           width=600, height=500, 
                                                           tools=['hover'],)*gvts.CartoLight*gv.Points([(st.lon,
                                                                                                         st.lat)]).opts(color='darkred',
                                                                                                                               width=500)#* gf.borders * gf.coastline#

In [ ]:
#Sources:
#http://xarray.pydata.org/en/stable/io.html
#https://poopcode.com/coronavirus-covid-19-live-tracking-dashboard-death-counts/
#https://stackoverflow.com/questions/17170229/setting-transparency-based-on-pixel-values-in-matplotlib
#https://data-dive.com/interactive-maps-made-easy-geoviews